# 用 ARIMA 做零售需求预测

读取与 Prophet 教程相同的商超、红酒、庭院三套 CSV，输出训练内回看、四个历史滚动起点与最终测试成绩。最后 28 天为 2025-12-04 至 2025-12-31。

从仓库根目录启动 Jupyter，并先安装 `pyproject.toml`，然后 Restart Kernel and Run All。`src/retail_forecast/common.py` 保存共同数据协议，各模型模块保存可检查的完整训练实现。所有输入 CSV 不改写。

训练内回看只描述拟合能力；历史窗口按业务生成机制预先指定，每个起点仅使用当时历史。评价排除闭店/缺货标签，所有方案共用日期和非负/营业日历处理。商超与红酒促销全年重复，因此促销观察窗口仍混合促销和非促销日，另外输出条件切片。


## 1. 数据、业务任务与评估协议

三个案例分别代表平滑加法需求、低销量计数需求与强季节需求。诊断真值 `conditional_mean`、`latent_demand` 不进入特征。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os,sys
def find_repo_root():
    configured=os.environ.get('RETAIL_FORECAST_ROOT')
    candidates=[Path(configured).expanduser()] if configured else [Path.cwd(),*Path.cwd().parents]
    for candidate in candidates:
        candidate=candidate.resolve()
        if (candidate/'pyproject.toml').is_file() and (candidate/'notebooks').is_dir():return candidate
    raise RuntimeError('找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT')
REPO_ROOT=find_repo_root()
sys.path.insert(0,str(REPO_ROOT/'src'))
import inspect,json
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Code
from retail_forecast.common import *
display(pd.DataFrame([{'scenario':n,'days':len(f),'cutoff':cutoff,'stockout_days':int(f.stockout.sum())} for n,f in datasets.items()]))
display(pd.DataFrame([{'scenario':n,**w} for n in SCENARIOS for w in windows(n)]))

## 2. ARIMA 与业务回归的数学关系

$$z_t=\Delta^d y_t,\qquad z_t=c+\sum_{i=1}^p\phi_i z_{t-i}+\sum_{j=1}^q\theta_j\varepsilon_{t-j}+\varepsilon_t.$$

变量：$y_t$ 是销量，$z_t$ 是差分后序列；$p,d,q$ 分别是 AR 阶数、差分次数、MA 阶数；$c$ 是常数，$\phi_i,\theta_j$ 是待估系数，$\varepsilon_t$ 是零均值新息；$\Delta$ 表示相邻差分，$\sum$ 表示求和。MA 使用历史新息。

业务回归采用 $y_t=\mathbf x_t^\top\boldsymbol\beta+u_t$，并令 $u_t$ 服从 ARIMA。$\mathbf x_t$ 是当天已知业务/日历特征，$\boldsymbol\beta$ 是回归系数，$u_t$ 是业务回归误差；$\top$ 表示转置。业务效应与 ARIMA 误差参数联合估计。

本例比较纯 ARIMA(1,1,1)、SARIMA(1,1,1)(1,0,0)[7]、同外生特征的独立误差回归和 ARIMA(1,0,1) 误差回归，配合每周 Seasonal naive。$d=0$ 的动态回归使用显式时间特征；这个选择仍需残差与未来窗口检验。


In [ ]:
scenario='grocery'
X=make_features(datasets[scenario],scenario)
display(X.head());print('Feature columns:',list(X));print('Truth excluded:',not {'y','latent_demand','conditional_mean'}.intersection(X))

## 3. 训练实现与参数检查

完整实现可在配套 Python 模块阅读。下面直接展示当前运行的训练函数，保证解释与代码一致。

In [ ]:
from retail_forecast.arima import *
display(Code(inspect.getsource(fit_arima),language="python"))

## 4. 实际训练、滚动验证与最终测试

每个历史窗口重新拟合；TFT 默认恢复此前在同一起点训练的检查点。树和神经网络的内部验证全部位于该起点之前。最终模型的历史回看参数已见过训练标签。

In [ ]:
m,p,details,summary,finals=run_family("arima",NAMES,fit_arima,predict_arima)
display(m.round(4));display(summary.round(4));display(details)

## 5. 预测与性能图

训练、滚动验证和最终测试来自不同时间段。比较 WAPE 时同时考虑销量分母，并查看 MAE、RMSE 和偏差。

In [ ]:
colors=plot_family("arima",NAMES,m,p,summary)

## 6. 参数与诊断

诊断用于发现结构遗漏，未来泛化由时间窗口评价。特征重要性与注意力都需要业务解释，不能直接作为因果效应。

In [ ]:
parameters,diagnostics=diagnose_arima(finals)
display(parameters);display(diagnostics)

## 7. 结果口径与系统交付

输出包括预测起点、目标日期、步长、原始预测和营业日历处理后的销售预测。补货系统还要结合提前期、库存和服务水平。当前脚本提供本地可重复实验；生产发布需补齐数据快照、权限、调度、质量门槛与回滚。

In [ ]:
out=RUNS_DIR/'arima'
assert len(m)==3*len(NAMES)*6
assert m.loc[m.split.eq('test'),'n'].nunique()==1
for name,digest in INPUT_HASHES.items():assert hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()==digest
print('Exports:',out.resolve())
display(summary.pivot(index='scenario',columns='model',values='WAPE'))